Analisis de `plan_de_compras_2025.xlsx`, mostrando la informacion segun fechas, tal cual como se encuentra en el archivo, sin normalizar la informacion en absoluto

In [ ]:
# Librerías necesarias:
# - pandas: para cargar y manipular la tabla del Excel como DataFrame
# - pathlib: para construir la ruta al archivo de forma independiente del sistema
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)  # mostrar todas las columnas al imprimir el DataFrame
from pathlib import Path

pd.set_option("display.max_columns", None)  # mostrar todas las columnas al imprimir el DataFrame

In [ ]:
# El archivo .xlsx lo busca en la carpeta raíz del proyecto"
RUTA_EXCEL = Path("..") / "plan_de_compras_2025.xlsx"

# Cargar la hoja "PLAN DE COMPRAS 2025"
df = pd.read_excel(RUTA_EXCEL, sheet_name=0)

print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
df.head()

In [ ]:
# Revisamos los tipos de dato que pandas asignó a cada columna.
# "Fecha de Inicio Compra" y "Fecha Publicación PAC 2025" deberían quedar como datetime64
# porque Excel las guarda con formato de fecha nativo.
# "Meses envío OC" queda como texto (object) porque mezcla varios formatos de fecha en un solo string.
df.dtypes

## 1. Búsqueda por rango de fechas

Filtramos el DataFrame quedándonos solo con las filas cuya `Fecha de Inicio Compra` cae dentro de un rango `[fecha_desde, fecha_hasta]`.

In [ ]:
def buscar_por_rango_de_fecha(df: pd.DataFrame, columna: str, fecha_desde: str, fecha_hasta: str) -> pd.DataFrame:
    """
    Devuelve las filas de `df` cuya fecha en `columna` está entre fecha_desde y fecha_hasta (ambas inclusive).
    fecha_desde / fecha_hasta se aceptan como texto, ej: "2023-01-01".
    """
    # pd.to_datetime convierte los strings de entrada a Timestamp para poder compararlos con la columna
    desde = pd.to_datetime(fecha_desde)
    hasta = pd.to_datetime(fecha_hasta)

    # .between() genera una máscara booleana fila por fila; la usamos para filtrar el DataFrame
    mascara = df[columna].between(desde, hasta)
    return df.loc[mascara].sort_values(columna)


# Ejemplo: proyectos cuya compra inició entre enero y marzo de 2023
resultado = buscar_por_rango_de_fecha(df, "Fecha de Inicio Compra", "2023-01-01", "2023-03-31")
resultado[["ID Proyecto", "Nombre Proyecto", "Fecha de Inicio Compra", "Fecha Publicación PAC 2025"]]

## 2. Búsqueda por año o por mes exacto

Muchas veces solo interesa "todo lo del año 2025" o "todo lo de enero 2025". Usamos los accesores `.dt.year` y `.dt.month` de pandas, que extraen el año/mes de cada fecha de la columna.

In [ ]:
def buscar_por_anio_mes(df: pd.DataFrame, columna: str, anio: int, mes: int | None = None) -> pd.DataFrame:
    """
    Devuelve las filas cuya fecha en `columna` cae en `anio` (y opcionalmente en `mes`, 1-12).
    """
    mascara = df[columna].dt.year == anio
    if mes is not None:
        mascara &= df[columna].dt.month == mes
    return df.loc[mascara].sort_values(columna)


# Ejemplo: publicaciones del PAC en noviembre de 2022
resultado_pac = buscar_por_anio_mes(df, "Fecha Publicación PAC 2025", anio=2022, mes=11)
print(f"Coincidencias: {len(resultado_pac)}")
resultado_pac[["ID Proyecto", "Nombre Proyecto", "Fecha Publicación PAC 2025"]].head(10)

## 3. Muestra de "Meses envío OC" tal como aparece en el archivo

Esta columna guarda una o varias fechas separadas por comas, pero el formato **cambia de fila en fila**: a veces es `"2023-04-01 00:00:00"`, a veces `"01-03-2023,01-03-2024,01-03-2025"` y a veces `"Ene 2023,Ene 2024,Ene 2025"`. Con esta mezcla no se puede filtrar ni ordenar de forma confiable la informacion.

In [ ]:
# .unique() lista los valores distintos que toma la columna; con esto se aprecia la mezcla de formatos
df["Meses envío OC"].dropna().sample(8, random_state=1).tolist()